# Phase 25 — revaluation: everything shipped since Phase 21, plus the calculator

Phase 21 (`scripts/23`) is the last full judge-scored run on `eval_v3`. Everything
below it has shipped without a judge pass: Phase 22 (ask-instead-of-assume, judgment
questions), Phase 23/23a (multi-turn — not testable here, `eval_v3` is single-turn by
construction), and all of Phase 24 (contribution/end-of-service/ceiling/gender
calculators, `top_k=5`, `FAMILY_TRIGGER_DEPTH`). Phase 24c/24d/24e's own trials were
3-6 manual reps each, never a harness run, and `eval_v3` itself contains no question
that states a salary, so it has never been able to measure the calculator at all.

## Two tracks, reported separately, never blended

**Track 1 — `eval_v3` regression (60 q).** Retrieval + generation + Claude-Sonnet-5
judge, fresh (not replayed, since retrieval changed since Phase 21), compared against
`eval/phase21_judge.csv` / `eval/phase21_generations.csv`.

**Track 2 — `eval/eval_calc_v4.jsonl` (8 q, new).** One question per real live bug
found this session: salary-stated contributions (the 5.25% fabrication), the ceiling
exception, end-of-service, gender conditions (Arabic single-turn, English, and the
age-only variant), and insurance names per component. Retrieval is checked once per
question (deterministic-ish); generation is repeated `N_TRIALS=4` times per question
at production temperature, since Phase 24's own findings were sampling-dependent (the
gender case was 5/6, not 6/6). Graded against `res["calculation"]` directly — the
exact structure the model was handed — plus a check that the final answer text
actually narrates it, since Phase 24e found the block being correct does not
guarantee the model states it correctly (the `MULTITURN_REMINDER` conflict). No LLM
judge on this track: a regex/structure check on "does 142.5 appear" is strictly more
precise than an LLM judge would be for this, and cheaper.

## A retrieval fix already landed before this run

Building the ground truth for Track 2 surfaced a live gap: two of the eight phrasings
did not retrieve the full contribution family at `top_k=5` (`FAMILY_TRIGGER_DEPTH`
was 35; the needed member sat at raw rank 36-37). Widened to 37, swept 35-40 against
`eval_v3` + `eval_rates.jsonl` + the two new probes, confirmed **zero** change to any
`eval_v3` retrieval metric. Already committed and pushed (`d668d32`) before this
notebook runs, so Track 1's retrieval numbers below already reflect it.

## Pre-committed reading

- **Track 1** is a regression check, not a comparison the way Phase 20→21 was one
  (that pair held retrieval fixed; retrieval changed here). A large faithfulness/
  relevancy drop would mean Phase 22's rules or the calculator's context injection
  are hurting ordinary answers. Flat-to-better means they aren't.
- **Track 2** has no prior baseline — Phase 24's own numbers were never run through
  a harness. This establishes the reference point for the calculator's true
  reliability under repeated sampling, the way Phase 20 did for generation overall.
- Judge noise floor is imported from Phase 12b (~0.1 on the 1-5 scale), not
  re-measured here — a single run per arm, per established project practice.

In [1]:
import json
import os
import re as _re
import sys
import time
from pathlib import Path

import pandas as pd

PROJECT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "retrieval.py").exists()), None)
assert PROJECT, "run this notebook from inside the project tree"
sys.path.insert(0, str(PROJECT))

from src.retrieval import Retriever, EMBED_PROVIDER, PINECONE_INDEX_NAME, RRF_W, RRF_K, FAMILY_TRIGGER_DEPTH, FAMILY_TRIGGER_COUNT
from src.generation import (get_production_generator, SYSTEM_PROMPT, ARTICLE_NUM_RE,
                            extract_cited_articles, grounded_article_numbers)

EVAL_V3 = PROJECT / "eval" / "eval_v3.jsonl"
EVAL_CALC = PROJECT / "eval" / "eval_calc_v4.jsonl"

RETR_PATH = PROJECT / "eval" / "phase25_v3_retrieval.csv"
GEN_PATH = PROJECT / "eval" / "phase25_v3_generations.csv"
JUDGE_PATH = PROJECT / "eval" / "phase25_v3_judge.csv"

BASE_GEN = PROJECT / "eval" / "phase21_generations.csv"
BASE_JUDGE = PROJECT / "eval" / "phase21_judge.csv"

CALC_RETR_PATH = PROJECT / "eval" / "phase25_calc_retrieval.csv"
CALC_GEN_PATH = PROJECT / "eval" / "phase25_calc_generations.csv"

ev = pd.DataFrame([json.loads(l) for l in open(EVAL_V3, encoding="utf-8")])
evc = pd.DataFrame([json.loads(l) for l in open(EVAL_CALC, encoding="utf-8")])

n_rules = len(_re.findall(r"^\d+\.", SYSTEM_PROMPT, _re.MULTILINE))
assert n_rules == 20, f"expected the 20-rule Phase 24e prompt, found {n_rules}"
assert FAMILY_TRIGGER_DEPTH == 37, f"expected the Phase 25 retrieval fix (depth 37), found {FAMILY_TRIGGER_DEPTH}"

print(f"eval_v3: {len(ev)} questions | eval_calc_v4: {len(evc)} questions")
print(f"retrieval: {EMBED_PROVIDER} -> {PINECONE_INDEX_NAME}, RRF k={RRF_K} w={RRF_W}, "
      f"family trigger depth={FAMILY_TRIGGER_DEPTH} count={FAMILY_TRIGGER_COUNT}")
print(f"system prompt: {n_rules} rules, {len(SYSTEM_PROMPT)} chars")

eval_v3: 60 questions | eval_calc_v4: 8 questions
retrieval: gemini -> jordan-labor-law-articles-gemini, RRF k=5 w=0.9, family trigger depth=37 count=2
system prompt: 20 rules, 6412 chars


## Track 1.1 — eval_v3 retrieval (fresh)

Same production `Retriever`, `top_k` from its own default (5).

In [2]:
if RETR_PATH.exists():
    retr = pd.read_csv(RETR_PATH, encoding="utf-8-sig")
    retr["chunk_ids"] = retr["chunk_ids"].apply(json.loads)
    print(f"loaded cached retrieval for {len(retr)} questions")
else:
    r = Retriever()
    rows = []
    for q in ev.itertuples(index=False):
        chunks = r.retrieve(q.question_text)
        want = {str(a) for a in q.answer_article_numbers}
        rank, found = None, set()
        for pos, c in enumerate(chunks, start=1):
            if (c["law"] == q.answer_law and c["language"] == q.answer_language
                    and c["article_number"] in want):
                if rank is None:
                    rank = pos
                found.add(c["article_number"])
        rows.append({"question_id": q.question_id, "chunk_ids": [c["chunk_id"] for c in chunks],
                     "rank": rank, "n_found": len(found), "full_coverage": found == want})
        print(f"  retrieved {len(rows)}/{len(ev)}", end="\r")
    retr = pd.DataFrame(rows)
    out = retr.copy()
    out["chunk_ids"] = out["chunk_ids"].apply(json.dumps)
    out.to_csv(RETR_PATH, index=False, encoding="utf-8-sig")
    print(f"\nsaved -> {RETR_PATH.name}")

chunks_tbl = pd.read_parquet(PROJECT / "0. cleaned_data" / "chunks_articles.parquet")
chunks_tbl["article_number"] = chunks_tbl["article_number"].astype(str)
BY_ID = {c.chunk_id: c for c in chunks_tbl.itertuples()}


def chunks_for(qid, retr_df):
    ids = retr_df.loc[retr_df.question_id == qid, "chunk_ids"].iloc[0]
    return [{"chunk_id": i, "law": BY_ID[i].law, "language": BY_ID[i].language,
             "article_number": BY_ID[i].article_number, "text": BY_ID[i].text} for i in ids]


r_ = retr["rank"]
N = len(retr)
print(f"recall@1 {(r_ == 1).sum()/N:.3f}  recall@5 {r_.notna().sum()/N:.3f}  "
      f"mrr {(1/r_.dropna()).sum()/N:.3f}  full_coverage {retr.full_coverage.sum()/N:.3f}  "
      f"worst_rank {r_.max():.0f}  misses {int(r_.isna().sum())}")

loaded cached retrieval for 60 questions
recall@1 0.883  recall@5 1.000  mrr 0.926  full_coverage 0.950  worst_rank 5  misses 0


## Track 1.2 — eval_v3 generation

Production generator (`gpt-4o-mini`), no history (single-turn, matches `eval_v3`'s design).

In [3]:
gen_rows = []
if GEN_PATH.exists():
    gen_rows = pd.read_csv(GEN_PATH, encoding="utf-8-sig").to_dict("records")
done = {r["question_id"] for r in gen_rows}
print(f"{len(done)} answers already saved, {len(ev) - len(done)} to generate")

if len(done) < len(ev):
    generator = get_production_generator()
    for q in ev.itertuples(index=False):
        if q.question_id in done:
            continue
        ch = chunks_for(q.question_id, retr)
        res = generator.generate(q.question_text, ch)
        gen_rows.append({
            "question_id": q.question_id, "question_text": q.question_text,
            "question_language": q.question_language, "answer": res["answer"],
            "citations": json.dumps(res["citations"], ensure_ascii=False),
            "calculation": json.dumps(res.get("calculation"), ensure_ascii=False),
        })
        pd.DataFrame(gen_rows).to_csv(GEN_PATH, index=False, encoding="utf-8-sig")
        print(f"  generated {len(gen_rows)}/{len(ev)}", end="\r")

gen = pd.DataFrame(gen_rows)
print(f"\n{len(gen)} answers ready")
print(f"answers where the calculator fired: {gen['calculation'].apply(lambda s: s not in (None, 'null')).sum()}/{len(gen)}")

60 answers already saved, 0 to generate

60 answers ready
answers where the calculator fired: 60/60


## Track 1.3 — regex proxy

Using the current (Phase 21+) definitions: `grounded_article_numbers` excludes in-text cross-references from the fabrication check, `extract_cited_articles` normalises Arabic-Indic digits.

In [4]:
rows = []
for q in ev.itertuples(index=False):
    ans = gen.loc[gen.question_id == q.question_id, "answer"].iloc[0]
    ch = chunks_for(q.question_id, retr)
    grounded = grounded_article_numbers(ch)
    want = {str(a) for a in q.answer_article_numbers}
    cited = extract_cited_articles(ans)
    rows.append({
        "question_id": q.question_id, "cited": sorted(cited),
        "correct_article_cited": bool(cited & want),
        "all_correct_articles_cited": want <= cited,
        "fabricated": sorted(cited - grounded), "has_fabrication": bool(cited - grounded),
        "answer_chars": len(str(ans)),
    })
proxy = pd.DataFrame(rows)
n = len(proxy)
print(f"correct article cited      : {proxy.correct_article_cited.sum()}/{n} ({proxy.correct_article_cited.mean():.1%})")
print(f"all correct articles cited : {proxy.all_correct_articles_cited.sum()}/{n}")
print(f"answers with a fabrication  : {proxy.has_fabrication.sum()}/{n}")
if proxy.has_fabrication.any():
    print(proxy[proxy.has_fabrication][["question_id", "cited", "fabricated"]].to_string(index=False))

correct article cited      : 59/60 (98.3%)
all correct articles cited : 55/60
answers with a fabrication  : 0/60


## Track 1.4 — LLM judge

Same judge, same prompt, same model as Phase 20/21, via `evallib` — the definition
verified byte-identical across every notebook that has ever used it.

In [5]:
from openai import OpenAI
from dotenv import load_dotenv

load_dotenv(PROJECT / ".env")
judge_client = OpenAI(api_key=os.environ["OPENROUTER_API_KEY"], base_url="https://openrouter.ai/api/v1")

from evallib import JUDGE_MODEL, JUDGE_PROMPT, JUDGE_MAX_TOKENS, make_judge
judge_one = make_judge(judge_client, max_tokens=JUDGE_MAX_TOKENS)

judge_rows = []
if JUDGE_PATH.exists():
    judge_rows = pd.read_csv(JUDGE_PATH, encoding="utf-8-sig").to_dict("records")
jdone = {r["question_id"] for r in judge_rows}
print(f"{len(jdone)} already judged, {len(ev) - len(jdone)} to go")

for q in ev.itertuples(index=False):
    if q.question_id in jdone:
        continue
    ans = gen.loc[gen.question_id == q.question_id, "answer"].iloc[0]
    verdict = judge_one(q.question_text, chunks_for(q.question_id, retr), ans)
    judge_rows.append({"question_id": q.question_id, **verdict})
    pd.DataFrame(judge_rows).to_csv(JUDGE_PATH, index=False, encoding="utf-8-sig")
    print(f"  judged {len(judge_rows)}/{len(ev)}", end="\r")

jdg = pd.DataFrame(judge_rows)
print(f"\n{len(jdg)} judged")

60 already judged, 0 to go

60 judged


## Track 1 results — vs Phase 21

In [6]:
full = ev.merge(retr, on="question_id").merge(proxy, on="question_id").merge(jdg, on="question_id")
full.to_csv(PROJECT / "eval" / "phase25_v3_summary.csv", index=False, encoding="utf-8-sig")

base_gen = pd.read_csv(BASE_GEN, encoding="utf-8-sig")
base_j = pd.read_csv(BASE_JUDGE, encoding="utf-8-sig")
cmp_ = base_j.merge(jdg, on="question_id", suffixes=("_21", "_25"))
cmp_ = cmp_.merge(ev[["question_id", "question_language", "answer_law"]], on="question_id")
cmp_.to_csv(PROJECT / "eval" / "phase25_v3_comparison.csv", index=False, encoding="utf-8-sig")

r_ = full["rank"]
N = len(full)
print("=" * 74)
print("RETRIEVAL")
print("=" * 74)
print(f"  recall@1 {(r_==1).sum()/N:.3f}  recall@5 {r_.notna().sum()/N:.3f}  "
      f"mrr {(1/r_.dropna()).sum()/N:.3f}  full_coverage {full.full_coverage.sum()/N:.3f}  "
      f"worst_rank {r_.max():.0f}  misses {int(r_.isna().sum())}")

print()
print("=" * 74)
print("GENERATION -- regex proxy")
print("=" * 74)
print(f"  correct article cited        {full.correct_article_cited.sum()}/{N}")
print(f"  all correct articles cited   {full.all_correct_articles_cited.sum()}/{N}")
print(f"  answers with a fabrication   {full.has_fabrication.sum()}/{N}")

print()
print("=" * 74)
print("GENERATION -- LLM judge, Phase 21 (12 rules, no calculator) vs Phase 25 (20 rules + calculator)")
print("=" * 74)
print(f"{'':16s} {'Phase 21':>10s} {'Phase 25':>10s} {'delta':>9s}   better/worse/same")
for axis in ["faithfulness", "relevancy"]:
    a, b = cmp_[f"{axis}_score_21"], cmp_[f"{axis}_score_25"]
    up, dn = (b > a).sum(), (b < a).sum()
    print(f"{axis:16s} {a.mean():>10.3f} {b.mean():>10.3f} {b.mean()-a.mean():>+9.3f}   "
          f"{up} / {dn} / {len(a)-up-dn}")

print("\nscore distribution")
for axis in ["faithfulness", "relevancy"]:
    for arm in ["21", "25"]:
        s = cmp_[f"{axis}_score_{arm}"]
        print(f"  {axis:13s} phase{arm}  " + "  ".join(f"{v}:{(s==v).sum():2d}" for v in [5,4,3,2,1]))

print("\nregressions -- worse in Phase 25")
worse = cmp_[(cmp_.faithfulness_score_25 < cmp_.faithfulness_score_21) |
             (cmp_.relevancy_score_25 < cmp_.relevancy_score_21)]
print(f"{len(worse)} of {N}")
for r in worse.itertuples():
    print(f"  {r.question_id} [{r.question_language}] faith {r.faithfulness_score_21}->{r.faithfulness_score_25}  "
          f"rel {r.relevancy_score_21}->{r.relevancy_score_25}")
    if r.faithfulness_score_25 < r.faithfulness_score_21:
        print(f"      faith: {r.faithfulness_reason_25}")
    if r.relevancy_score_25 < r.relevancy_score_21:
        print(f"      rel  : {r.relevancy_reason_25}")

RETRIEVAL
  recall@1 0.883  recall@5 1.000  mrr 0.926  full_coverage 0.950  worst_rank 5  misses 0

GENERATION -- regex proxy
  correct article cited        59/60
  all correct articles cited   55/60
  answers with a fabrication   0/60

GENERATION -- LLM judge, Phase 21 (12 rules, no calculator) vs Phase 25 (20 rules + calculator)
                   Phase 21   Phase 25     delta   better/worse/same
faithfulness          4.800      4.700    -0.100   6 / 10 / 44
relevancy             4.800      4.717    -0.083   6 / 9 / 45

score distribution
  faithfulness  phase21  5:49  4:10  3: 1  2: 0  1: 0
  faithfulness  phase25  5:45  4:12  3: 3  2: 0  1: 0
  relevancy     phase21  5:50  4: 8  3: 2  2: 0  1: 0
  relevancy     phase25  5:47  4: 9  3: 4  2: 0  1: 0

regressions -- worse in Phase 25
15 of 60
  q02 [en] faith 5->4  rel 5->5
      faith: The claims about transport expenses and prosthetic devices are directly supported by Article 26, though the answer omits that this coverage applies s

## Track 2 — the calculator, under repeated sampling

8 new questions, one per real live bug. Retrieval is checked once per question
(deterministic-ish); generation runs `N_TRIALS` times per question at production
temperature, since the model narrating a correct computed block correctly is a
separate question from the block being correct (Phase 24e: the gender block was
present in 6/6 trials but the model only stated both figures in 2/6 before a prompt
fix).

In [7]:
N_TRIALS = 4

if CALC_RETR_PATH.exists():
    cretr = pd.read_csv(CALC_RETR_PATH, encoding="utf-8-sig")
    cretr["chunk_ids"] = cretr["chunk_ids"].apply(json.loads)
else:
    # A fresh Retriever every time, deliberately -- not "reuse `r` if the name is
    # already bound". A later cell's `for r in worse.itertuples(): ...` leaves a
    # pandas row object named `r` sitting in globals after Track 1, and a
    # dir()-based reuse check picked that up instead of the Retriever instance,
    # crashing with AttributeError: 'Pandas' object has no attribute 'retrieve'.
    # Building a new Retriever here is cheap (parquet + BM25 + Pinecone client,
    # no embedding call yet) so there is no reason to share the instance at all.
    calc_retriever = Retriever()
    rows = []
    for q in evc.itertuples(index=False):
        chunks = calc_retriever.retrieve(q.question_text)
        want = {str(a) for a in q.answer_article_numbers}
        found = {c["article_number"] for c in chunks if c["law"] == q.answer_law
                 and c["language"] == q.answer_language and c["article_number"] in want}
        rows.append({"question_id": q.question_id, "chunk_ids": [c["chunk_id"] for c in chunks],
                     "full_coverage": found == want, "n_found": len(found)})
    cretr = pd.DataFrame(rows)
    out = cretr.copy()
    out["chunk_ids"] = out["chunk_ids"].apply(json.dumps)
    out.to_csv(CALC_RETR_PATH, index=False, encoding="utf-8-sig")

print(f"retrieval full coverage: {cretr.full_coverage.sum()}/{len(cretr)}")
print(cretr[["question_id", "full_coverage", "n_found"]].to_string(index=False))

def calc_chunks_for(qid):
    ids = cretr.loc[cretr.question_id == qid, "chunk_ids"].iloc[0]
    return [{"chunk_id": i, "law": BY_ID[i].law, "language": BY_ID[i].language,
             "article_number": BY_ID[i].article_number, "text": BY_ID[i].text} for i in ids]

retrieval full coverage: 8/8
question_id  full_coverage  n_found
        c01           True        4
        c02           True        2
        c03           True        5
        c04           True        1
        c05           True        2
        c06           True        2
        c07           True        2
        c08           True        4


In [8]:
calc_rows = []
if CALC_GEN_PATH.exists():
    calc_rows = pd.read_csv(CALC_GEN_PATH, encoding="utf-8-sig").to_dict("records")
done_trials = {(r["question_id"], r["trial"]) for r in calc_rows}
todo = [(q.question_id, t) for q in evc.itertuples(index=False) for t in range(1, N_TRIALS + 1)
        if (q.question_id, t) not in done_trials]
print(f"{len(done_trials)} trials already saved, {len(todo)} to go")

if todo:
    # Built unconditionally, same reasoning as calc_retriever above: a dir()-based
    # "reuse if already bound" check is fragile against unrelated loop-variable
    # leakage elsewhere in the notebook, and building a generator is cheap.
    calc_generator = get_production_generator()
    for qid, trial in todo:
        q = evc[evc.question_id == qid].iloc[0]
        ch = calc_chunks_for(qid)
        res = calc_generator.generate(q.question_text, ch)
        calc_rows.append({
            "question_id": qid, "trial": trial, "question_text": q.question_text,
            "question_language": q.question_language, "question_category": q.question_category,
            "answer": res["answer"],
            "calculation": json.dumps(res.get("calculation"), ensure_ascii=False),
        })
        pd.DataFrame(calc_rows).to_csv(CALC_GEN_PATH, index=False, encoding="utf-8-sig")
        print(f"  generated {len(calc_rows)}/{len(evc) * N_TRIALS}", end="\r")

calc_gen = pd.DataFrame(calc_rows)
print(f"\n{len(calc_gen)} trials ready")

0 trials already saved, 32 to go


  generated 32/32
32 trials ready


## Track 2 — grading

Graded against `res["calculation"]` directly (the deterministic structure
`compute_contributions`/`compute_end_of_service`/`compute_gender_conditions`
actually returned), plus a check that the number/behaviour it required is present
in the *narrated* answer text -- the block being correct does not guarantee the
model states it (Phase 24e's `MULTITURN_REMINDER` conflict). Every check is
question-specific and written from `src/calc.py`'s actual return schema, not
inferred.

In [9]:
import ast

def get_calc(row):
    raw = row["calculation"]
    if pd.isna(raw) or raw in (None, "null"):
        return None
    return json.loads(raw)


def get_contrib(calc):
    if calc is None:
        return None
    if "employer_rate" in calc:      # flat shape: only compute_contributions fired
        return calc
    return calc.get("contributions")  # nested shape: contributions + eos and/or gender


def get_eos(calc):
    return calc.get("end_of_service") if calc else None


def get_gender(calc):
    return calc.get("gender_conditions") if calc else None


AR_DIGITS = str.maketrans("٠١٢٣٤٥٦٧٨٩", "0123456789")

def has_number(text, value, tol=0.01):
    """value appears in the answer text, allowing both 'x' and 'x.0'/'x,0' formatting."""
    t = str(text).translate(AR_DIGITS)
    for cand in (f"{value:g}", f"{value:.1f}", f"{value:.2f}", f"{int(value)}" if value == int(value) else None):
        if cand and cand in t:
            return True
    return False


def check_c01(calc, ans):
    c = get_contrib(calc)
    calc_ok = bool(c and abs(c["employer_rate"] - 14.25) < 0.01 and c.get("employer_amount") == 142.5)
    ans_ok = has_number(ans, 142.5) or has_number(ans, 14.25)
    return calc_ok, ans_ok


def check_c02(calc, ans):
    c = get_contrib(calc)
    calc_ok = bool(c and abs(c["employee_rate"] - 7.5) < 0.01 and c.get("employee_amount") == 75.0)
    ans_ok = has_number(ans, 75.0) or has_number(ans, 7.5)
    return calc_ok, ans_ok


def check_c03(calc, ans):
    c = get_contrib(calc)
    calc_ok = bool(c and c.get("capped") and c.get("ceiling_exception_applies")
                   and c.get("ceiling") == 3000.0 and c.get("ceiling_exception") == 5000.0)
    # must surface the ask -- a bare date substring is not enough, require the digits together
    ans_ok = ("15" in ans and "10" in ans and "2009" in ans) or ("5000" in ans.replace(",", "")) or ("5,000" in ans)
    return calc_ok, ans_ok


def check_c04(calc, ans):
    e = get_eos(calc)
    calc_ok = bool(e and e.get("months_owed") == 10 and e.get("amount") == 8000.0 and e.get("requires_not_ssl"))
    # must raise the SSL precondition as a question/condition, not state 8000 flatly as owed
    asks = any(kw in ans for kw in ["الضمان الاجتماعي", "Social Security", "خاضع", "subject"])
    ans_ok = has_number(ans, 8000.0) and asks
    return calc_ok, ans_ok


def check_gender(calc, ans, expect_sex=None):
    g = get_gender(calc)
    have_62 = any(x["article"] == "62" for x in (g or []))
    have_64 = any(x["article"] == "64" for x in (g or []))
    calc_ok = bool(g and have_62 and have_64
                   and next(x for x in g if x["article"] == "62")["male"] == 60
                   and next(x for x in g if x["article"] == "62")["female"] == 55
                   and next(x for x in g if x["article"] == "64")["male"] == 252
                   and next(x for x in g if x["article"] == "64")["female"] == 228)
    both = (has_number(ans, 60) or has_number(ans, 52)) and (has_number(ans, 55) or has_number(ans, 28))
    asked = ("؟" in ans and ("ذكر" in ans or "male" in ans.lower())) or ("?" in ans and "male" in ans.lower())
    silently_one = not both and not asked
    return calc_ok, (both or asked) and not silently_one


def check_c08(calc, ans):
    c = get_contrib(calc)
    names = [comp.get("name") for comp in (c or {}).get("components", [])] if c else []
    calc_ok = bool(c and all(names) and len(names) == 6)
    markers = ["إصاب", "أمومة", "تعطل", "شيخوخة", "Injury", "Maternity", "Unemployment", "Old-Age", "Old Age"]
    ans_ok = sum(1 for m in markers if m in ans) >= 3   # at least 3 of the 4 insurance types named
    return calc_ok, ans_ok


CHECKS = {
    "c01": check_c01, "c02": check_c02, "c03": check_c03, "c04": check_c04,
    "c05": lambda calc, ans: check_gender(calc, ans), "c06": lambda calc, ans: check_gender(calc, ans),
    "c07": lambda calc, ans: check_gender(calc, ans), "c08": check_c08,
}

results = []
for row in calc_gen.itertuples(index=False):
    calc = get_calc(dict(row._asdict()))
    calc_ok, ans_ok = CHECKS[row.question_id](calc, str(row.answer))
    results.append({"question_id": row.question_id, "trial": row.trial,
                     "calc_correct": calc_ok, "narration_correct": ans_ok})

rdf = pd.DataFrame(results)
rdf.to_csv(PROJECT / "eval" / "phase25_calc_grading.csv", index=False, encoding="utf-8-sig")

print("=" * 74)
print(f"TRACK 2 -- calculator, {N_TRIALS} trials per question")
print("=" * 74)
summary = rdf.groupby("question_id").agg(
    calc_correct=("calc_correct", "sum"), narration_correct=("narration_correct", "sum"),
    n=("trial", "count"))
print(summary.to_string())
print(f"\ncalc structure correct (should be ~deterministic): {rdf.calc_correct.sum()}/{len(rdf)}")
print(f"narration correct (sampling-dependent):              {rdf.narration_correct.sum()}/{len(rdf)}")

TRACK 2 -- calculator, 4 trials per question
             calc_correct  narration_correct  n
question_id                                    
c01                     4                  4  4
c02                     4                  4  4
c03                     4                  4  4
c04                     0                  3  4
c05                     4                  4  4
c06                     4                  3  4
c07                     4                  3  4
c08                     4                  4  4

calc structure correct (should be ~deterministic): 28/32
narration correct (sampling-dependent):              29/32


## Verdict

### Track 1 — eval_v3 (60 q), retrieval

recall@1 **0.883**, recall@5 **1.000**, MRR **0.926**, full coverage **0.950**,
worst rank **5** (Phase 24c reported 4; recall@5 stayed 1.000 either way, so this
does not cross the boundary that matters — attributed to Pinecone's approximate-NN,
already documented as a source of small variation, not a regression). The
`FAMILY_TRIGGER_DEPTH` 35→37 fix carried zero cost, as swept.

### Track 1 — generation, regex proxy

Correct article cited **59/60 (98.3%)**, all correct articles cited **55/60**,
fabrications **0/60**.

### Track 1 — generation, LLM judge vs Phase 21

| axis | Phase 21 | Phase 25 | delta | better/worse/same | sign test |
|---|---|---|---|---|---|
| faithfulness | 4.800 | 4.700 | −0.100 | 6/10/44 | **p = 0.45 — noise** |
| relevancy | 4.800 | 4.717 | −0.083 | 6/9/45 | **p = 0.61 — noise** |

Both deltas sit at or inside the Phase 12b noise floor and neither sign test is
close to significant. **Read as held, not regressed** — same standard Phase 21
applied to its own +0.100 faithfulness move. This is not a clean comparison the
way Phase 20→21 was (retrieval changed here), so "held" is the most that can be
claimed either way.

**Correction to the cell 5 printout above:** "calculator fired 60/60" is wrong —
it's a bug in that one print expression (`s not in (None, 'null')` doesn't call
`pd.isna` first, so a `'null'` string that round-tripped through `to_csv`/`read_csv`
came back as NaN and satisfied the condition by accident). The true count, read
directly from the saved answers: **6/60** — 4 `gender_conditions`, 2 `contributions`,
0 `end_of_service`. Does not affect retrieval/proxy/judge scoring above, which never
reads this column.

**A real design gap, found by that correction:** `compute_gender_conditions` has no
completeness/relevance guard — unlike contributions (requires the whole funding
family) or end-of-service (requires Article 32 AND a stated duration), it fires
whenever SSL Article 62 or 64 merely appears in the retrieved context, whether or
not the question is actually about sex-differentiated pension eligibility. It fired
on 4/60 eval_v3 questions (q14, q15, q27, q28 — the other two calculator hits, q10
and q16, were `contributions`), none of which ask about that. **2 of the 4
gender-fired questions (q15, q27) are in the 15-question regression list** —
plausible contributing factor, not proven: the judge's stated reasons for both are
sub-clause precision/omission (64(b) vs 64(a)(2) for q15; 63(a) vs 63(c) for q27),
consistent with either the injected block crowding attention or ordinary retrieval
crowding from the same siblings. Not fixed here — flagged for a decision.

### Track 2 — calculator (8 q × 4 trials), calc-structure correctness

c01/c02/c03/c05/c08: **4/4**. c06/c07: **4/4** (gender block extracted correctly
every time). **c04: 0/4** — not a defect in `compute_end_of_service` itself, but
`extract_service` only parses digit-form years ("10 سنوات"), not the spelled-out
word form this question used ("عشر سنوات"). Confirmed directly: `extract_service("...عشر سنوات...")`
returns `None`. A real, previously-undocumented gap — `_AGE_WORDS_AR` already solved
exactly this for ages in Phase 24e, but years-of-service was never given the same
treatment.

### Track 2 — narration correctness (does the answer actually say it)

c01/c02/c03/c05/c08: **4/4**. c06: 3/4, c07: 3/4 — each failing trial is a genuine
silent-male-default (discusses only the male threshold, 252 or "if male", never
mentions the female figure at all), consistent with Phase 24e's own disclosed ~1-in-6
residual rate on this exact failure mode — not a new regression, the same known
probabilistic ceiling.

**c04's proxy-reported 3/4 overstates it.** The `ans_ok` check only tested for the
*keywords* "الضمان الاجتماعي"/"خاضع"/"Social Security"/"subject" appearing anywhere,
which Article 32's own retrieved text contains regardless of what the model does with
it. Reading all 4 trials directly: trials 1–3 quote the "not subject to Social
Security" precondition verbatim and then immediately state 8,000 JD as what "you are
entitled to" without ever asking or conditioning the figure on it; trial 4 omits the
precondition entirely. **The true narration success rate is 0/4** — every trial
reproduces exactly the pre-Phase-24a failure mode (a confident wage×years answer for
a majority-inapplicable provision), because the calculator never engaged for this
phrasing. This is the most important finding in this run: the deterministic guard
that Phase 24a built is silently bypassed for a natural, plausible way to state years
of service, and the fallback behavior is the original bug, not a safe default.

### Two follow-ups, not executed here (need a decision, not a judgment call)

1. Add Arabic word-number support to `extract_service` (mirror `_AGE_WORDS_AR`) so
   "عشر سنوات" parses the same as "10 سنوات" — closes the real gap above.
2. Add a completeness/relevance guard to `compute_gender_conditions`, the same shape
   as the other two calculators, so it stops firing on ordinary questions where
   Article 62/64 is present only as retrieval crowding.

### What this run does not establish

No comparison arm for Track 2 (Phase 24's own trials were never run through a
harness, so there is no prior number to compare against — this **is** the reference
point going forward). Track 1's judge comparison is confounded by retrieval having
changed since Phase 21, unlike the clean Phase 20→21 pair. Single run per arm
throughout; Track 1's noise floor is imported from Phase 12b, not re-measured.